# FXS / 02: confirmation 2021–2025 of the frozen candidates

Protocol: [`001_claude_fxs01_preregistration.md`](../../discussions/FX_Intraday_Seasonality/001_claude_fxs01_preregistration.md) §5. Reads `01/candidates_frozen.csv` unchanged (its SHA-256 must match `01/summary.json`).

Each candidate is traded every day at quoted prices (long: buy ask at start, sell bid at end; short: sell bid, buy ask). Commission 0.70 bp round trip.

| Gate | Condition |
|---|---|
| K1 | mean after spread > 0, NW one-sided p, BH across candidates q ≤ 0.05 |
| K2 | net: block-bootstrap lower bound > 0 at blocks 10 and 20 |
| K3 | mean net at 2× commission > 0 |
| K4 | yearly mean net > 0 in ≥ 3 of 5 years |

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs_")
    os.environ["FXS_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
PAIRS = list(F.PAIRS)
if SYNTHETIC:
    PAIRS = ["EURUSD", "USDJPY", "CADCHF"]
    cfg = replace(cfg, n_placebo=3, n_boot=2_000)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120)
IN = Path.cwd() / ("01_synthetic" if SYNTHETIC else "01")
OUT = Path.cwd() / ("02_synthetic" if SYNTHETIC else "02"); OUT.mkdir(exist_ok=True)
START, END = F.CONFIRMATION
frozen = IN / "candidates_frozen.csv"
disc = json.loads((IN / "summary.json").read_text())
sha = F.file_sha256(frozen)
assert sha == disc["candidates_sha256"], "candidates_frozen.csv changed after discovery"
cand = pd.read_csv(frozen)
print(f"{len(cand)} frozen candidates (sha256 {sha[:12]}...) | discovery D0 = {disc['D0']} | confirmation {START} -> {END}")
display(cand[["id", "pair", "clock", "start_time", "end_time", "direction", "mean_bps", "t"]])

## 1. Confirmation panels (2021–2025, candidate pairs only)

In [ ]:
need = sorted(set(cand["pair"]))
panels = F.load_panels(need, START, END, "confirmation", cfg)
display(pd.DataFrame([{"pair": p, "clock": c, "days": len(v.dates), "median_spread_bps": float(np.median(v.spread_bps()))}
                      for (p, c), v in panels.items()]).round(3))

## 2. Gates K1–K4

In [ ]:
trades = {r["id"]: F.candidate_trades(panels[(r["pair"], r["clock"])], int(r["start"]), int(r["end"]), int(r["direction"]), cfg)
          for _, r in cand.iterrows()}
res = cand[["id", "pair", "clock", "start_time", "end_time", "direction"]].assign(disc_mean_bps=cand["mean_bps"]).merge(
    F.confirm(trades, cfg), on="id") if len(cand) else pd.DataFrame()
if len(res):
    display(res.drop(columns=["boot_net_lower"]).round(3))
    res.to_csv(OUT / "02_confirmation.csv", index=False)
    print(res["verdict"].value_counts().to_string())

## 3. By year and equity per candidate

Yearly mean net (bps) and the cumulative net return of one unit traded every day, in bps.

In [ ]:
if len(res):
    yearly = pd.DataFrame({cid: tr.groupby("year")["net_bps"].mean() for cid, tr in trades.items()}).T
    display(yearly.round(2)); yearly.to_csv(OUT / "02_by_year_net.csv")
    n = len(res)
    fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(16, 3 * np.ceil(n / 3)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, res.iterrows()):
        tr = trades[r["id"]]
        ax.plot(pd.to_datetime(tr["date"]), tr["net_bps"].cumsum(), lw=0.9, color="C2" if r["verdict"] == "CONFIRMED" else "C3")
        ax.axhline(0, color="black", lw=0.5)
        ax.set_title(f"{r['id']} {r['pair']} {r['start_time']}-{r['end_time'].split()[1]}: {r['verdict']}", fontsize=9)
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "02_equity.png", dpi=110); plt.show()

## 4. Discovery vs confirmation

If the discovery means were mostly selection noise, confirmation means collapse towards zero (points near the horizontal axis).

In [ ]:
if len(res):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(res["direction"] * res["disc_mean_bps"], res["gross_bps"],
               c=np.where(res["verdict"] == "CONFIRMED", "C2", "C3"))
    lim = max(1.0, float(np.nanmax(np.abs(res["direction"] * res["disc_mean_bps"]))))
    ax.plot([0, lim], [0, lim], "k--", lw=0.6); ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("discovery mean in trade direction (bps, mid)"); ax.set_ylabel("confirmation mean after spread (bps)")
    plt.tight_layout(); plt.savefig(OUT / "02_shrinkage.png", dpi=110); plt.show()
    print("median ratio confirmation / discovery:", float(np.median(res["gross_bps"] / (res["direction"] * res["disc_mean_bps"]))))

In [ ]:
summary = {"study": "FXS02", "fxs_version": F.FXS_VERSION, "synthetic": SYNTHETIC, "window": [str(START), str(END)],
           "candidates_sha256": sha, "discovery_D0": disc["D0"], "n_candidates": int(len(cand)),
           "confirmed": res.loc[res["verdict"] == "CONFIRMED", "id"].tolist() if len(res) else [],
           "results": res.to_dict(orient="records") if len(res) else []}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("discovery_D0", "n_candidates", "confirmed")}, indent=2))